# Week 78, Day 3: Scheduling jobs

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w078d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from datetime import datetime, timedelta

start = datetime(2026, 11, 10, 9, 0)
interval = timedelta(minutes=15)
times = [start + interval * n for n in range(4)]
print([t.strftime("%H:%M") for t in times])

late = datetime(2026, 11, 10, 9, 17, 40)
print((late + interval).strftime("%H:%M:%S"))
missed = (late - start) // interval
print(missed, (start + interval * (missed + 1)).strftime("%H:%M"))

In [ ]:
from datetime import datetime, timedelta

def every(field_value, n):
    return field_value % n == 0

# "*/20 9-10 * * 1-5": every 20 minutes from 09:00 to 10:40 on weekdays
def matches(t):
    cron_weekday = (t.weekday() + 1) % 7      # Python: Monday is 0; cron: Sunday is 0
    return every(t.minute, 20) and 9 <= t.hour <= 10 and 1 <= cron_weekday <= 5

t = datetime(2026, 11, 13, 8, 50)            # a Friday
runs = []
while len(runs) < 7:
    t += timedelta(minutes=1)
    if matches(t):
        runs.append(t.strftime("%a %H:%M"))
print(runs)

In [ ]:
class FakeClock:
    def __init__(self, t=0):
        self.t = t
    def __call__(self):
        return self.t

clock = FakeClock(1000)
jobs = [{"name": "poll", "every": 60, "due": 1000}]

def run_pending():
    now, ran = clock(), []
    for job in jobs:
        if job["due"] <= now:
            ran.append(job["name"])
            missed = (now - job["due"]) // job["every"]
            job["due"] += (missed + 1) * job["every"]     # on the grid, skipping missed slots
    return ran

for t in (1000, 1030, 1075, 1250, 1260):
    clock.t = t
    print(t, run_pending(), "next", jobs[0]["due"])

> **Key idea.** Cron expressions have five fields (minute, hour, day of month, month, day of week with 0 and 7 as Sunday); items are *, values, ranges a-b, steps */n and a-b/n, and comma lists; a time matches when every field matches, except that when both day fields are restricted, either may match. Shortcuts: @hourly, @daily, @weekly, @monthly. Python's weekday() has Monday as 0, cron has Sunday as 0. Cron uses the machine's time zone (GitHub Actions uses UTC); DST can skip or repeat small-hours jobs. A scheduler keeps jobs with due times, reads an injected clock, reschedules on the grid (scheduled time plus interval) and runs missed slots once. Unattended jobs must be idempotent, lock against overlaps (open(path, "x"), released in finally, stale locks taken over), retry with capped exponential backoff and jitter, log with timestamps, and fail loudly (exit codes, alerts, a dead man's switch); under cron, use absolute paths.

### Exercise: Parse a cron field

Write `parse_field(text, low, high)`: the set of values a cron field allows, for a field whose values run from `low` to `high`. Items are separated by commas; each is `*`, a number, a range `a-b`, or either of those followed by a step `/n` (`*/15`, `10-40/10`, and `5/20` meaning from 5 to `high` in steps of 20). Raise `ValueError(f"bad cron field: {text}")` for anything malformed: a value outside `low` to `high`, a range that goes backwards, a step of `0`, an empty item, or text that isn't a number. Then `parse_cron(expr)`: split the expression into its five fields and return a list of five sets (minute 0 to 59, hour 0 to 23, day 1 to 31, month 1 to 12, weekday 0 to 7, with `7` turned into `0`), raising `ValueError(f"expected 5 fields: {expr}")` for the wrong number of fields.

In [ ]:
def parse_field(text, low, high):
    ___

def parse_cron(expr):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_field():\n    \"every item form\"\n    assert parse_field(\"*\", 0, 5) == {0, 1, 2, 3, 4, 5}\n    assert parse_field(\"*/15\", 0, 59) == {0, 15, 30, 45}\n    assert parse_field(\"10-40/10\", 0, 59) == {10, 20, 30, 40}\n    assert parse_field(\"5/20\", 0, 59) == {5, 25, 45}\n    assert parse_field(\"1,15,30-31\", 1, 31) == {1, 15, 30, 31}\n    assert parse_field(\"1-5\", 0, 7) == {1, 2, 3, 4, 5}\n    for bad in (\"60\", \"5-1\", \"*/0\", \"1,,2\", \"a\", \"\", \"-3\", \"1-\"):\n        try:\n            parse_field(bad, 0, 59)\n        except ValueError as e:\n            assert str(e) == f\"bad cron field: {bad}\", bad\n        else:\n            raise AssertionError(f\"{bad!r} should be refused\")\n\ndef test_cron():\n    \"five fields, Sunday as 0 or 7\"\n    m, h, dom, mon, dow = parse_cron(\"30 2 * * 7\")\n    assert m == {30} and h == {2} and dom == set(range(1, 32)) and mon == set(range(1, 13)) and dow == {0}\n    assert parse_cron(\"0 9 * * 1-5\")[4] == {1, 2, 3, 4, 5} and parse_cron(\"0 0 * * *\")[4] == set(range(7))\n    try:\n        parse_cron(\"0 9 * *\")\n    except ValueError as e:\n        assert str(e) == \"expected 5 fields: 0 9 * *\"\n    else:\n        raise AssertionError(\"four fields\")")

### Exercise: Does it run now?

Write `matches(expr, when)`: `True` if the cron expression matches the `datetime` `when` (to the minute; seconds are ignored). Use your `parse_field` (include it again) and remember both rules: cron's Sunday is `0` (Python's `weekday()` has Monday as `0`), and when **both** day fields are restricted (neither is exactly `*`), the day matches if **either** matches. Accept the shortcuts `@hourly`, `@daily`, `@weekly`, `@monthly` and `@yearly` too.

In [ ]:
from datetime import datetime

SHORTCUTS = {"@hourly": "0 * * * *", "@daily": "0 0 * * *", "@weekly": "0 0 * * 0",
             "@monthly": "0 0 1 * *", "@yearly": "0 0 1 1 *"}

def parse_field(text, low, high):
    ___

def matches(expr, when):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_matches():\n    \"fields and weekdays\"\n    from datetime import datetime\n    mon = datetime(2026, 11, 9, 9, 0, 42)         # a Monday\n    assert matches(\"0 9 * * 1-5\", mon) and not matches(\"0 9 * * 0,6\", mon)\n    assert matches(\"*/15 * * * *\", datetime(2026, 11, 9, 9, 45)) and not matches(\"*/15 * * * *\", datetime(2026, 11, 9, 9, 50))\n    sun = datetime(2026, 11, 15, 2, 30)\n    assert matches(\"30 2 * * 0\", sun) and matches(\"30 2 * * 7\", sun) and not matches(\"30 2 * * 1\", sun)\n    assert matches(\"0 0 1 * *\", datetime(2026, 12, 1)) and not matches(\"0 0 1 * *\", datetime(2026, 12, 2))\n\ndef test_or_rule():\n    \"either day field, when both are restricted\"\n    from datetime import datetime\n    assert matches(\"0 9 1 * 1\", datetime(2026, 11, 9, 9))       # a Monday, not the 1st\n    assert matches(\"0 9 1 * 1\", datetime(2026, 10, 1, 9))       # the 1st, a Thursday\n    assert not matches(\"0 9 1 * 1\", datetime(2026, 11, 10, 9))  # neither\n    assert not matches(\"0 9 * * 1\", datetime(2026, 10, 1, 9))   # day of month is *: weekday decides\n\ndef test_shortcuts():\n    \"@daily and friends\"\n    from datetime import datetime\n    assert matches(\"@daily\", datetime(2026, 5, 4)) and not matches(\"@daily\", datetime(2026, 5, 4, 1))\n    assert matches(\"@weekly\", datetime(2026, 11, 15)) and matches(\"@yearly\", datetime(2027, 1, 1))\n    assert matches(\"@hourly\", datetime(2026, 5, 4, 13)) and matches(\"@monthly\", datetime(2026, 2, 1))")

### Exercise: When does it next run?

Using `matches` (include it and `parse_field`), write `next_runs(expr, after, n)`: the next `n` times strictly after the `datetime` `after` that match, as a list of `datetime`s with zero seconds. Step minute by minute, but skip whole days quickly: when a day can't match at all (check the day at midnight with the hour and minute fields ignored), jump straight to the next midnight. Raise `ValueError("no run within a year")` if no match turns up within 366 days of the starting point.

In [ ]:
from datetime import datetime, timedelta

SHORTCUTS = {"@hourly": "0 * * * *", "@daily": "0 0 * * *", "@weekly": "0 0 * * 0",
             "@monthly": "0 0 1 * *", "@yearly": "0 0 1 1 *"}

def parse_field(text, low, high):
    ___

def matches(expr, when):
    ___

def next_runs(expr, after, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_next():\n    \"the next few runs\"\n    from datetime import datetime\n    got = next_runs(\"0 9 * * 1-5\", datetime(2026, 11, 13, 9, 0), 3)          # from Friday 09:00\n    assert got == [datetime(2026, 11, 16, 9), datetime(2026, 11, 17, 9), datetime(2026, 11, 18, 9)]\n    assert next_runs(\"*/20 * * * *\", datetime(2026, 1, 1, 23, 45, 10), 2) == [datetime(2026, 1, 2, 0, 0), datetime(2026, 1, 2, 0, 20)]\n    assert next_runs(\"@monthly\", datetime(2026, 12, 15), 2) == [datetime(2027, 1, 1), datetime(2027, 2, 1)]\n    assert next_runs(\"0 12 29 2 *\", datetime(2027, 3, 1), 1) == [datetime(2028, 2, 29, 12)]\n\ndef test_never():\n    \"an impossible schedule\"\n    from datetime import datetime\n    try:\n        next_runs(\"0 0 31 2 *\", datetime(2026, 1, 1), 1)\n    except ValueError as e:\n        assert str(e) == \"no run within a year\"\n    else:\n        raise AssertionError(\"February 31st never comes\")\n\ndef test_fast():\n    \"days are skipped, not walked minute by minute\"\n    import time\n    from datetime import datetime\n    start = time.perf_counter()\n    assert next_runs(\"0 0 1 1 *\", datetime(2026, 1, 1, 0, 1), 1) == [datetime(2027, 1, 1)]\n    assert time.perf_counter() - start < 3, \"too slow: skip days that can't match\"")

### Exercise: A scheduler with a fake clock

Write a class `Scheduler(clock)`, where `clock()` returns the time in seconds. `every(seconds, func, name)` adds a job, first due `seconds` after the current time. `run_pending()` runs every job that is due (due time at or before `clock()`), in order of due time and then name; a job that raises is logged as failed but doesn't stop the others. After each run, the job's next due time moves **along its grid**: the old due time plus the interval, as many times as needed to be after now (missed slots aren't run again). It returns a list of `(name, "ok")` or `(name, "failed: MESSAGE")` pairs for the jobs it ran. `next_due()` returns the earliest due time, or `None` if there are no jobs.

In [ ]:
class Scheduler:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_grid():\n    \"on time, no drift, missed slots skipped\"\n    class Clock:\n        t = 1000\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    s = Scheduler(clock)\n    ran = []\n    s.every(60, lambda: ran.append(\"poll\"), \"poll\")\n    s.every(300, lambda: ran.append(\"report\"), \"report\")\n    assert s.next_due() == 1060 and s.run_pending() == []\n    clock.t = 1075\n    assert s.run_pending() == [(\"poll\", \"ok\")] and s.next_due() == 1120\n    clock.t = 1300\n    assert s.run_pending() == [(\"poll\", \"ok\"), (\"report\", \"ok\")]\n    assert s.next_due() == 1360\n    clock.t = 1599\n    assert s.run_pending() == [(\"poll\", \"ok\")] and s.next_due() == 1600\n    clock.t = 1600\n    assert s.run_pending() == [(\"poll\", \"ok\"), (\"report\", \"ok\")]\n    assert ran == [\"poll\", \"poll\", \"report\", \"poll\", \"poll\", \"report\"]\n\ndef test_failures():\n    \"a failing job doesn't stop the others\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    s = Scheduler(clock)\n    def broken():\n        raise ConnectionError(\"server down\")\n    s.every(10, broken, \"b-sync\")\n    s.every(10, lambda: None, \"a-clean\")\n    clock.t = 10\n    assert s.run_pending() == [(\"a-clean\", \"ok\"), (\"b-sync\", \"failed: server down\")]\n    assert Scheduler(clock).next_due() is None")

### Exercise: One run at a time

Write `run_locked(lock_path, job, now, stale_after=3600)`: run `job()` only if no other run holds the lock. Create the lock file with mode `"x"` (which fails with `FileExistsError` if it's there) and write `now` into it. If it already exists and holds a time less than `stale_after` seconds before `now`, return `"skipped"` without running; if it's older (or unreadable), it's stale: delete it and take the lock. Always remove the lock when the job finishes, even if the job raises (let the exception propagate). Return `"ran"` after a successful run.

In [ ]:
import os
from pathlib import Path

def run_locked(lock_path, job, now, stale_after=3600):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lock():\n    \"skips while locked, cleans up after\"\n    from pathlib import Path\n    lock = Path(\"job.lock\")\n    lock.unlink(missing_ok=True)\n    seen = []\n    def job():\n        seen.append(lock.read_text())\n        assert run_locked(lock, lambda: seen.append(\"inner\"), 1001) == \"skipped\"\n    assert run_locked(lock, job, 1000) == \"ran\"\n    assert seen == [\"1000\"] and not lock.exists()\n\ndef test_stale_and_errors():\n    \"stale locks are taken over; the lock is released after a crash\"\n    from pathlib import Path\n    lock = Path(\"job.lock\")\n    lock.write_text(\"100\")\n    assert run_locked(lock, lambda: None, 5000) == \"ran\" and not lock.exists()\n    lock.write_text(\"4000\")\n    assert run_locked(lock, lambda: None, 5000) == \"skipped\" and lock.exists()\n    lock.write_text(\"garbage\")\n    assert run_locked(lock, lambda: None, 5000) == \"ran\"\n    def boom():\n        raise RuntimeError(\"disk full\")\n    try:\n        run_locked(lock, boom, 6000)\n    except RuntimeError:\n        pass\n    else:\n        raise AssertionError(\"the error should propagate\")\n    assert not lock.exists()")

### Exercise: Retry with backoff

Write `retry(func, attempts=4, base=1.0, cap=30.0, sleep=time.sleep, retry_on=(OSError,))`: call `func()` and return its result. If it raises one of the `retry_on` exceptions, wait and try again, up to `attempts` calls in all; the waits are `base`, `base*2`, `base*4`... each capped at `cap`, and they're passed to `sleep` (so tests can record them). After the last attempt, re-raise the last exception. Other exceptions are raised at once, without retrying.

In [ ]:
import time

def retry(func, attempts=4, base=1.0, cap=30.0, sleep=time.sleep, retry_on=(OSError,)):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_retry():\n    \"backoff, success and giving up\"\n    calls, waits = [], []\n    def flaky():\n        calls.append(1)\n        if len(calls) < 3:\n            raise ConnectionError(\"blip\")\n        return \"done\"\n    assert retry(flaky, sleep=waits.append) == \"done\" and waits == [1.0, 2.0] and len(calls) == 3\n    waits = []\n    def down():\n        raise TimeoutError(\"still down\")\n    try:\n        retry(down, attempts=6, base=5, cap=30, sleep=waits.append)\n    except TimeoutError as e:\n        assert str(e) == \"still down\"\n    else:\n        raise AssertionError(\"should give up\")\n    assert waits == [5, 10, 20, 30, 30]\n\ndef test_no_retry():\n    \"only the listed errors are retried\"\n    waits = []\n    def bad():\n        raise ValueError(\"bad input\")\n    try:\n        retry(bad, sleep=waits.append)\n    except ValueError:\n        pass\n    assert waits == []")

### Bug hunt: A job that drifted

This scheduler sets the next run from **when the job finished**, so each slow run pushes the schedule later, and the four runs come at `['09:00', '09:15:20', '09:30:40', '09:46:00']`. Keep it on the 15-minute grid, so it prints `['09:00', '09:15', '09:30', '09:45']`.

In [ ]:
from datetime import datetime, timedelta

interval = timedelta(minutes=15)
due = datetime(2026, 11, 10, 9, 0)
starts = []
for _ in range(4):
    starts.append(due)
    finished = due + timedelta(seconds=20)     # each run takes 20 seconds
    due = finished + interval
print([t.strftime("%H:%M" if t.second == 0 else "%H:%M:%S") for t in starts])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"scheduled time plus the interval\"\n    expect_output(\"['09:00', '09:15', '09:30', '09:45']\")")

### Bug hunt: Mondays at the wrong time

The report should run at 09:00 every weekday, but the check converts Python's weekday straight into cron's, where Sunday is `0`, so Monday counts as `0` (Sunday) and Saturday as `5`, and this prints `['Tue', 'Wed', 'Thu', 'Fri', 'Sat']`. Convert properly, so it prints `['Mon', 'Tue', 'Wed', 'Thu', 'Fri']`.

In [ ]:
from datetime import datetime, timedelta

WEEKDAYS = {1, 2, 3, 4, 5}       # cron's Monday to Friday
day = datetime(2026, 11, 9, 9, 0)   # a Monday
runs = []
for n in range(7):
    t = day + timedelta(days=n)
    if t.weekday() in WEEKDAYS:
        runs.append(t.strftime("%a"))
print(runs)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"Sunday is 0 in cron\"\n    expect_output(\"['Mon', 'Tue', 'Wed', 'Thu', 'Fri']\")")

### Bug hunt: A lock that never let go

When the job fails, the lock file is never removed, so every later run is skipped and this prints `ran, failed, skipped, skipped`. Release the lock however the job ends, so it prints `ran, failed, ran, ran`.

In [ ]:
from pathlib import Path

LOCK = Path("nightly.lock")
LOCK.unlink(missing_ok=True)

def run(job):
    try:
        LOCK.touch(exist_ok=False)
    except FileExistsError:
        return "skipped"
    try:
        job()
    except Exception:
        return "failed"
    LOCK.unlink()
    return "ran"

def fails():
    raise OSError("disk full")

print(", ".join([run(lambda: None), run(fails), run(lambda: None), run(lambda: None)]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"finally\"\n    expect_output(\"ran, failed, ran, ran\")")

### Puzzle: A job that looks after itself

Arrange a nightly job that takes a lock, logs its start and end, reports failure with an exit code, and always releases the lock. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the hours in the cron field "*/6" for hours (a sorted list)
q = ___   # the cron day of week for a datetime whose weekday() is 6
r = ___   # the expression for 02:30 every day
s = ___   # True or False: "0 9 1 * 1" runs on a Monday the 9th
t = ___   # the waits, capped at 10, for 5 attempts with base 2 (a list)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [0, 6, 12, 18] and q == 0 and r == \"30 2 * * *\", \"p, q and r\"\n    assert s is True and t == [2, 4, 8, 10], \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a crontab runner

Write `simulate(crontab, start, end, jobs)`. `crontab` is the text of a crontab: blank lines and lines starting with `#` are ignored; every other line is a schedule (five fields or a shortcut such as `@daily`) followed by a job name, such as `30 2 * * * backup`. `jobs` maps names to functions that take the run time (a `datetime`) and return nothing or raise. Simulate every minute from `start` up to and including `end` (both on whole minutes): at each minute, run every line that matches, in crontab order. Return the log as a list of strings: `YYYY-MM-DD HH:MM NAME ok` or `YYYY-MM-DD HH:MM NAME failed: MESSAGE`. Before running anything, check the whole crontab: a line with a bad schedule raises `ValueError(f"line N: bad schedule")`, and a line naming a job that isn't in `jobs` raises `ValueError(f"line N: unknown job NAME")` (N counts every line from 1). A job that has failed three runs in a row is disabled for the rest of the simulation: log `... NAME disabled after 3 failures` once, straight after its third failure, and skip it afterwards. To stay fast, skip days that no line can match.

In [ ]:
from datetime import datetime, timedelta

SHORTCUTS = {"@hourly": "0 * * * *", "@daily": "0 0 * * *", "@weekly": "0 0 * * 0",
             "@monthly": "0 0 1 * *", "@yearly": "0 0 1 1 *"}

def simulate(crontab, start, end, jobs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_runs():\n    \"a day of jobs\"\n    from datetime import datetime\n    seen = []\n    jobs = {\"backup\": lambda t: seen.append((\"backup\", t.hour)),\n            \"poll\": lambda t: None,\n            \"report\": lambda t: seen.append((\"report\", t.strftime(\"%a\")))}\n    tab = \"\"\"\n# nightly backup\n30 2 * * * backup\n\n*/30 9-10 * * 1-5 poll\n@weekly report\n\"\"\"\n    log = simulate(tab, datetime(2026, 11, 14, 0, 0), datetime(2026, 11, 16, 9, 30), jobs)\n    assert log == [\"2026-11-14 02:30 backup ok\", \"2026-11-15 00:00 report ok\", \"2026-11-15 02:30 backup ok\",\n                   \"2026-11-16 02:30 backup ok\", \"2026-11-16 09:00 poll ok\", \"2026-11-16 09:30 poll ok\"]\n    assert seen == [(\"backup\", 2), (\"report\", \"Sun\"), (\"backup\", 2), (\"backup\", 2)]\n\ndef test_failures():\n    \"three failures in a row disable a job\"\n    from datetime import datetime\n    count = {\"n\": 0}\n    def flaky(t):\n        count[\"n\"] += 1\n        if count[\"n\"] in (1, 3, 4, 5):\n            raise ConnectionError(\"no route\")\n    log = simulate(\"*/10 * * * * sync\", datetime(2026, 1, 1, 0, 0), datetime(2026, 1, 1, 1, 30), {\"sync\": flaky})\n    assert log == [\"2026-01-01 00:00 sync failed: no route\", \"2026-01-01 00:10 sync ok\",\n                   \"2026-01-01 00:20 sync failed: no route\", \"2026-01-01 00:30 sync failed: no route\",\n                   \"2026-01-01 00:40 sync failed: no route\", \"2026-01-01 00:40 sync disabled after 3 failures\"]\n\ndef test_checks():\n    \"the crontab is checked before anything runs\"\n    from datetime import datetime\n    ran = []\n    for tab, message in ((\"0 0 * * * a\\n61 * * * * a\", \"line 2: bad schedule\"), (\"# hi\\n0 0 * * * nope\", \"line 2: unknown job nope\"),\n                         (\"0 0 * * a\", \"line 1: bad schedule\"), (\"@daily\", \"line 1: bad schedule\")):\n        try:\n            simulate(tab, datetime(2026, 1, 1), datetime(2026, 1, 2), {\"a\": ran.append})\n        except ValueError as e:\n            assert str(e) == message, (tab, str(e))\n        else:\n            raise AssertionError(tab)\n    assert ran == []\n\ndef test_fast():\n    \"a whole year of a monthly job\"\n    import time\n    from datetime import datetime\n    started = time.perf_counter()\n    log = simulate(\"0 3 1 * * a\", datetime(2026, 1, 1), datetime(2026, 12, 31, 23, 59), {\"a\": lambda t: None})\n    assert len(log) == 12 and log[-1] == \"2026-12-01 03:00 a ok\"\n    assert time.perf_counter() - started < 5, \"skip days that can't match\"")

## Warm-up

Two ways to schedule "every 15 minutes": from **when the job last ran** (which creeps later every time a run is slow) or from **the original grid** (09:00, 09:15, 09:30...). Good schedulers use the grid. Floor division of one `timedelta` by another counts the whole intervals that fit, which is how a scheduler works out how many runs it missed.

## Cron expressions

**cron** is the Unix job scheduler, and its five-field expressions are the common language of scheduling: Linux and macOS crontabs, GitHub Actions (`on: schedule: - cron: "30 2 * * *"`), Kubernetes CronJobs and every cloud scheduler use them.

```
┌───────── minute        0-59
│ ┌─────── hour          0-23
│ │ ┌───── day of month  1-31
│ │ │ ┌─── month         1-12
│ │ │ │ ┌─ day of week   0-6 (0 is Sunday; 7 is Sunday too)
│ │ │ │ │
30 2 * * *     every day at 02:30
```

Each field is a list of values, separated by commas, where each item is:

| Item | Means | Example (in the minute field) |
|---|---|---|
| `*` | every value | every minute |
| `5` | that value | at minute 5 |
| `1-5` | a range, inclusive | minutes 1 to 5 |
| `*/15` | every 15th value from the start | 0, 15, 30, 45 |
| `10-40/10` | every 10th within the range | 10, 20, 30, 40 |
| `0,30` | a list | 0 and 30 |

A time matches when **every** field matches, with one famous exception: when **both** day of month and day of week are restricted (neither is `*`), a day matches if **either** does. So `0 9 1 * 1` means "09:00 on the 1st of every month **and** every Monday", not "on Mondays that fall on the 1st".

| Expression | Runs |
|---|---|
| `*/5 * * * *` | every five minutes |
| `0 * * * *` | on the hour, every hour |
| `0 9 * * 1-5` | 09:00 on weekdays |
| `30 2 * * 0` | 02:30 every Sunday |
| `0 0 1 * *` | midnight on the 1st of each month |
| `0 */6 * * *` | 00:00, 06:00, 12:00, 18:00 |
| `15 14 1,15 * *` | 14:15 on the 1st and 15th |

Many schedulers also accept shortcuts: `@hourly` (`0 * * * *`), `@daily` (`0 0 * * *`), `@weekly` (`0 0 * * 0`), `@monthly` (`0 0 1 * *`) and `@yearly`. **Time zones** matter: cron uses the machine's time zone (GitHub Actions uses UTC), and daylight-saving changes mean a 02:30 job is skipped on the night the clocks go forward and may run twice when they go back. Schedule important jobs in UTC, or away from the small hours.

Note the weekday conversion: Python's `weekday()` counts from **Monday = 0**, cron from **Sunday = 0**. Mixing them up is the classic cron bug.

## Schedulers in Python

Cron starts a **new process** for each run. Sometimes you want a long-running program that runs its own jobs instead (a bot, a small service, a script that should poll every minute while it's open). The parts of any scheduler:

- a list of **jobs**, each with its next **due time**;
- a loop that asks "what's due now?", runs it, works out its next due time, and sleeps until the next one;
- the **clock** it reads, which should be a parameter (`clock=time.time`), so tests can pass a fake clock and check a whole day of runs in a millisecond.

Two decisions every scheduler makes:

- **No drift**: the next due time is the **scheduled** time plus the interval, not "now" plus the interval; otherwise each slow run pushes every later run back.
- **Missed runs**: if the program was asleep (or a job took an hour) and three runs were missed, run the job **once** and skip ahead to the next future slot, rather than running it three times in a row. (Some jobs need every run made up; most don't.)

The standard library has `sched` (a small event scheduler driven by any clock and sleep function you pass). Third-party libraries add friendlier interfaces: **`schedule`** (`schedule.every(10).minutes.do(job)`) and **APScheduler** (cron-style triggers, persistent job stores). They all work the way you'll build today.

## Jobs that run alone

A job that runs unattended has nobody watching it, so it has to look after itself:

- **Idempotent**: running it twice (because of a retry, a catch-up or an overlap) must do no harm. Write "make sure X is true" jobs (sync these files, send emails **not already sent**, store the last processed id), not "do X again" jobs.
- **No overlaps**: if a run takes longer than the interval, the next one starts while it's still going. Take a **lock** at the start (a lock file created with `open(path, "x")`, which fails if it already exists), skip the run if it's taken, and **always** release it in a `finally`. Record the time in the lock so a lock left behind by a crashed run can be recognised as **stale** and taken over.
- **Retries with backoff**: a network blip shouldn't fail the night's backup, but hammering a struggling server makes things worse. Retry a few times, waiting longer each time: **exponential backoff** (1, 2, 4, 8 seconds), with a cap, and ideally a little random **jitter** so a thousand clients don't retry in step.
- **Logs with timestamps**: when it went wrong at 03:00, the log is all you have. Log the start, the end, how long it took, what it did, and the full error.
- **Loud failures**: exit with a non-zero code, and make sure someone is told: an email (cron mails a job's output to its owner if mail is set up), a chat message, or a **dead man's switch** (the job pings a monitoring service on success, and the service alerts you when the ping **stops** arriving, which also catches a job that never ran at all).
- **Cron's bare environment**: cron runs jobs with a minimal `PATH`, from your home folder, without your shell's settings. Use absolute paths for everything (the Python interpreter, the script, the files), and set what the job needs explicitly.

## Where schedules live

### On your own computer

- **Linux and macOS**: `crontab -e` edits your crontab; each line is an expression and a command, such as `30 2 * * * /home/ada/venv/bin/python /home/ada/backup.py >> /home/ada/backup.log 2>&1`. On Linux, **systemd timers** do the same with better logging (`journalctl`) and catch-up of missed runs (`Persistent=true`).
- **Windows**: Task Scheduler (the app, or `schtasks` on the command line).
- **GitHub Actions**: a workflow with `on: schedule: - cron: "0 6 * * 1"` runs in GitHub's cloud (in UTC, and not to the exact minute), with no server of your own; good for reports, link checks and data refreshes.

Check expressions before trusting them: sites such as crontab.guru explain one in words, and the third-party `croniter` library computes next run times in Python.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the schedules your own computer already runs (`crontab -l` and `systemctl list-timers` on Linux, Task Scheduler on Windows) and read a few of them with your new eyes. Tomorrow: **web scraping, responsibly**: reading data out of web pages, and doing it in a way that respects the people who run the site.